# Notebook 06 - Perceptron Classifier

This notebook trains a Perceptron classifier as a simple linear baseline for multiclass intrusion detection. It uses the same processed splits and evaluation style as the other models.

## 1. Imports

Load the libraries for data loading, Perceptron training, tuning, metrics, and saving artifacts.

The code below imports the Perceptron model, tuning tools, metrics, and artifact-saving utilities.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from pathlib import Path

from sklearn.linear_model import Perceptron
from sklearn.metrics import (
    f1_score,
    classification_report,
    confusion_matrix,
    recall_score,
    precision_score,
    accuracy_score
)
from sklearn.model_selection import GridSearchCV, PredefinedSplit
from sklearn.pipeline import Pipeline

## 2. Load Preprocessed Data

Load the feature variants and labels created in Notebook 2.

The code below loads the processed feature variants and encoded labels from Notebook 2 outputs.

In [ ]:
DATA_DIR = Path('../data/processed')
print('Using processed folder:', DATA_DIR.resolve())


def load_csv(name, base_dir=DATA_DIR):
    path = base_dir / name
    if not path.exists():
        raise FileNotFoundError(f'File not found: {path}')
    return pd.read_csv(path)


# --- Feature splits ---
X_train_raw        = load_csv('X_train_raw.csv')
X_val_raw          = load_csv('X_val_raw.csv')
X_test_raw         = load_csv('X_test_raw.csv')

X_train_out        = load_csv('X_train_out.csv')
X_val_out          = load_csv('X_val_out.csv')
X_test_out         = load_csv('X_test_out.csv')

X_train_scaled     = load_csv('X_train_scaled.csv')
X_val_scaled       = load_csv('X_val_scaled.csv')
X_test_scaled      = load_csv('X_test_scaled.csv')

X_train_out_scaled = load_csv('X_train_out_scaled.csv')
X_val_out_scaled   = load_csv('X_val_out_scaled.csv')
X_test_out_scaled  = load_csv('X_test_out_scaled.csv')

# --- Label splits ---
y_train_df = load_csv('y_train_encoded.csv')
y_val_df   = load_csv('y_val_encoded.csv')
y_test_df  = load_csv('y_test_encoded.csv')

for name, df in [('y_train', y_train_df), ('y_val', y_val_df), ('y_test', y_test_df)]:
    if df.shape[1] != 1:
        raise ValueError(f'{name}_encoded.csv must contain exactly one column.')

y_train = y_train_df.iloc[:, 0].to_numpy()
y_val   = y_val_df.iloc[:, 0].to_numpy()
y_test  = y_test_df.iloc[:, 0].to_numpy()

# --- Convert features to numpy ---
X_train_raw        = X_train_raw.to_numpy()
X_val_raw          = X_val_raw.to_numpy()
X_test_raw         = X_test_raw.to_numpy()

X_train_out        = X_train_out.to_numpy()
X_val_out          = X_val_out.to_numpy()
X_test_out         = X_test_out.to_numpy()

X_train_scaled     = X_train_scaled.to_numpy()
X_val_scaled       = X_val_scaled.to_numpy()
X_test_scaled      = X_test_scaled.to_numpy()

X_train_out_scaled = X_train_out_scaled.to_numpy()
X_val_out_scaled   = X_val_out_scaled.to_numpy()
X_test_out_scaled  = X_test_out_scaled.to_numpy()

# --- Organize into a dictionary ---
splits = {
    'raw':        {'X_train': X_train_raw,        'X_val': X_val_raw,        'X_test': X_test_raw},
    'out':        {'X_train': X_train_out,        'X_val': X_val_out,        'X_test': X_test_out},
    'scaled':     {'X_train': X_train_scaled,     'X_val': X_val_scaled,     'X_test': X_test_scaled},
    'out_scaled': {'X_train': X_train_out_scaled, 'X_val': X_val_out_scaled, 'X_test': X_test_out_scaled},
}

print('Data loaded successfully.')
print(f'Train: {y_train.shape[0]} samples | Val: {y_val.shape[0]} | Test: {y_test.shape[0]}')
print(f'Features: {X_train_raw.shape[1]} | Classes: {len(np.unique(y_train))}')

## 3. Select Feature Version

The Perceptron is sensitive to feature scale, so scaled variants are the most useful choices.

The code below selects the feature version used for this Perceptron run.

In [ ]:
# Options: 'raw' | 'out' | 'scaled' | 'out_scaled'
FEATURE_VERSION = 'out_scaled'

X_train = splits[FEATURE_VERSION]['X_train']
X_val   = splits[FEATURE_VERSION]['X_val']
X_test  = splits[FEATURE_VERSION]['X_test']
# y arrays are identical across all variants — no change needed

print(f'Feature version selected: {FEATURE_VERSION}')
print(f'X_train: {X_train.shape} | X_val: {X_val.shape} | X_test: {X_test.shape}')
print(f'y_train: {y_train.shape} | y_val: {y_val.shape} | y_test: {y_test.shape}')
print('\nClass distribution in y_train:')
print(pd.Series(y_train).value_counts().sort_index())
print('\nUnique classes:', np.unique(y_train))

## 4. Class Weights and Configuration

Use class weights to reduce the effect of imbalance. This helps rare classes contribute more during training.

The code below loads class weights and sets reproducible training options.

In [ ]:
custom_class_weights = {
    '0':  12.325989503608135, '1':  0.757872268907563,  '2':  6.815398101686718,
    '3':  7.3579831932773105, '4':  0.757872268907563,  '5':  1.3730991516598356,
    '6':  0.757872268907563,  '7':  0.757872268907563,  '8':  0.757872268907563,
    '9':  0.757872268907563,  '10': 1.6917870662519743, '11': 0.757872268907563,
    '12': 0.757872268907563,  '13': 0.757872268907563,  '14': 0.757872268907563,
    '15': 0.757872268907563,  '16': 3.0266464413241336, '17': 0.757872268907563,
    '18': 0.757872268907563,  '19': 0.757872268907563,  '20': 0.757872268907563,
    '21': 0.757872268907563,  '22': 0.757872268907563,  '23': 0.757872268907563,
    '24': 0.757872268907563,  '25': 0.757872268907563,  '26': 0.757872268907563,
    '27': 0.757872268907563,  '28': 17.531744488938998, '29': 0.757872268907563,
    '30': 7.546381055978579,  '31': 31.69119404034015,  '32': 0.757872268907563,
    '33': 10.229668110977519
}
custom_class_weights = {int(k): float(v) for k, v in custom_class_weights.items()}

BENIGN_LABEL = 1  # BENIGN = class index 1 in Notebook 02's alphabetical LabelEncoder

print(f'Class weights: {len(custom_class_weights)} classes')
print(f'Min weight: {min(custom_class_weights.values()):.4f}  (majority classes)')
print(f'Max weight: {max(custom_class_weights.values()):.4f}  (rarest class)')
print(f'Benign label: {BENIGN_LABEL}')

## 5. Fixed Validation Split

Use `PredefinedSplit` so tuning uses the same validation set as the other notebooks.

The code below combines train/validation data while preserving the fixed validation fold.

In [ ]:
X_train_val = np.vstack([X_train, X_val])
y_train_val = np.concatenate([y_train, y_val])

# -1 = always in training fold, 0 = always in validation fold
test_fold = np.concatenate([
    -1 * np.ones(len(X_train), dtype=int),
     0 * np.ones(len(X_val),   dtype=int)
])

predefined_split = PredefinedSplit(test_fold=test_fold)

print(f'X_train_val shape: {X_train_val.shape}')
print(f'Training samples   (fold = -1): {(test_fold == -1).sum()}')
print(f'Validation samples (fold =  0): {(test_fold == 0).sum()}')

## 6. Model and Hyperparameter Grid

Define the Perceptron and the small grid of regularization and training settings to test.

The code below defines the Perceptron and the hyperparameter grid.

In [ ]:
base_model = Perceptron(
    max_iter=1000,               # epochs over the training data
    tol=1e-3,                    # convergence tolerance
    shuffle=True,                # shuffle training data each epoch
    class_weight=custom_class_weights,
    n_jobs=-1,                   # parallelise the 34 OvR binary classifiers
    random_state=42
)

pipeline = Pipeline([('clf', base_model)])

param_grid = {
    'clf__penalty': ['l2', 'elasticnet'],   # regularization type
    'clf__alpha':   [1e-4, 1e-3, 1e-2],     # regularization strength
}

n_combos = 1
for v in param_grid.values():
    n_combos *= len(v)

print('Base model:', base_model)
print('Hyperparameter grid:', param_grid)
print(f'Total combinations: {n_combos}')

## 7. Hyperparameter Tuning

Tune with macro-F1 so each class has equal weight in the score. This matters because the dataset is imbalanced.

The code below runs grid search using macro-F1 and stores the best settings.

In [ ]:
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring='f1_macro',
    cv=predefined_split,
    refit=False,
    return_train_score=True,
    n_jobs=-1,
    verbose=2
)

grid_search.fit(X_train_val, y_train_val)

results = pd.DataFrame(grid_search.cv_results_).sort_values(
    by='mean_test_score', ascending=False
)

print('\nValidation tuning results:')
print(results[['param_clf__penalty', 'param_clf__alpha',
               'mean_train_score', 'mean_test_score', 'rank_test_score']].to_string())

best_penalty = str(results.iloc[0]['param_clf__penalty'])
best_alpha   = float(results.iloc[0]['param_clf__alpha'])
print(f'\nBest penalty selected: {best_penalty}')
print(f'Best alpha selected  : {best_alpha}')

## 8. Evaluation Function

Compute the same core metrics as the other notebooks. Since Perceptron does not provide probabilities, probability-based metrics are not used here.

The code below evaluates the model using hard predictions, since Perceptron has no probability output.

In [ ]:
N_CLASSES  = len(np.unique(y_train))
ALL_LABELS = np.arange(N_CLASSES)


def evaluate_intrusion_model(model, X, y, benign_label, split_name='Split'):
    y_pred = model.predict(X)

    macro_f1   = f1_score(y, y_pred, average='macro',  labels=ALL_LABELS, zero_division=0)
    macro_prec = precision_score(y, y_pred, average='macro', labels=ALL_LABELS, zero_division=0)
    macro_rec  = recall_score(y, y_pred, average='macro',  labels=ALL_LABELS, zero_division=0)
    accuracy   = accuracy_score(y, y_pred)

    per_class_recall = recall_score(y, y_pred, average=None, labels=ALL_LABELS, zero_division=0)
    cm = confusion_matrix(y, y_pred, labels=ALL_LABELS)

    benign_mask = (y == benign_label)
    benign_fpr  = np.mean(y_pred[benign_mask] != benign_label) if benign_mask.sum() > 0 else np.nan

    print(f'\n===== {split_name} =====')
    print(f'  Accuracy:        {accuracy:.6f}')
    print(f'  Macro-F1:        {macro_f1:.6f}')
    print(f'  Macro Precision: {macro_prec:.6f}')
    print(f'  Macro Recall:    {macro_rec:.6f}')
    print(f'  Benign FPR:      {benign_fpr:.6f}')
    print('\n  Per-class Recall:')
    for label, rec in zip(ALL_LABELS, per_class_recall):
        print(f'    Class {label:2d}: {rec:.4f}')
    print('\n  Classification Report:')
    print(classification_report(y, y_pred, labels=ALL_LABELS, zero_division=0))
    print('  Confusion Matrix:')
    print(cm)

    return {
        'split': split_name, 'accuracy': accuracy,
        'macro_f1': macro_f1, 'macro_prec': macro_prec, 'macro_rec': macro_rec,
        'benign_fpr': benign_fpr, 'per_class_recall': per_class_recall,
        'confusion_matrix': cm,
    }

## 9. Train vs Validation Check

Train the selected settings on `X_train` and compare train and validation scores to check generalization.

The code below trains on training data only and compares train vs validation results.

In [ ]:
best_model_train_only = Perceptron(
    max_iter=1000,
    tol=1e-3,
    shuffle=True,
    penalty=best_penalty,
    alpha=best_alpha,
    class_weight=custom_class_weights,
    n_jobs=-1,
    random_state=42
)
best_model_train_only.fit(X_train, y_train)

train_metrics = evaluate_intrusion_model(
    best_model_train_only, X_train, y_train, BENIGN_LABEL, 'Train'
)
val_metrics = evaluate_intrusion_model(
    best_model_train_only, X_val, y_val, BENIGN_LABEL, 'Validation'
)

# --- Grid search results bar chart ---
results_plot = results.sort_values(['param_clf__penalty', 'param_clf__alpha']).copy()
labels = [
    f"{row['param_clf__penalty']}\nalpha={row['param_clf__alpha']:.0e}"
    for _, row in results_plot.iterrows()
]
x = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - width/2, results_plot['mean_train_score'], width, label='Train Macro-F1', color='steelblue')
ax.bar(x + width/2, results_plot['mean_test_score'],  width, label='Val Macro-F1',   color='coral')
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel('Macro-F1')
ax.set_title('Perceptron — Train vs. Validation Macro-F1 by Hyperparameter Combination')
ax.legend()
ax.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.show()

## 10. Final Model

Retrain on train plus validation data and evaluate once on the test set.

The code below retrains on train+validation and evaluates the final model on test data.

In [ ]:
final_model = Perceptron(
    max_iter=1000,
    tol=1e-3,
    shuffle=True,
    penalty=best_penalty,
    alpha=best_alpha,
    class_weight=custom_class_weights,
    n_jobs=-1,
    random_state=42
)
final_model.fit(X_train_val, y_train_val)

# Convergence check
print(f'Iterations used: {final_model.n_iter_} / {final_model.max_iter}')
if final_model.n_iter_ >= final_model.max_iter:
    print('WARNING: model may not have fully converged — consider increasing max_iter.')
else:
    print('Model converged successfully.')
print(f'Weight matrix shape: {final_model.coef_.shape}  (n_classes x n_features)')

final_test_metrics = evaluate_intrusion_model(
    final_model, X_test, y_test, BENIGN_LABEL,
    'Final Test (retrained on train+val)'
)

## 11. Save Final Model

Save the trained Perceptron with `joblib`, using a filename that includes the preprocessing variant.

The code below saves the fitted Perceptron model under `data/artifacts/Perceptron/`.

In [ ]:
save_dir  = Path('../data/artifacts/Perceptron')
save_dir.mkdir(parents=True, exist_ok=True)
save_path = save_dir / f'perceptron_{FEATURE_VERSION}.joblib'
joblib.dump(final_model, save_path)
print(f'Saved final model to: {save_path}')

## 12. Save Metrics

Save the run metrics to JSON. For this model, scaled variants are the most meaningful.

The code below saves metrics and confusion matrix files for this run.

In [ ]:
import json

n_iter_val = final_model.n_iter_
if hasattr(n_iter_val, '__iter__'):
    n_iter_val = int(np.max(list(n_iter_val)))
else:
    n_iter_val = int(n_iter_val)

metrics_record = {
    'model':                'perceptron',
    'variant':              FEATURE_VERSION,
    'best_params':          {'penalty': best_penalty, 'alpha': best_alpha},
    'n_iter':               n_iter_val,
    'n_classes':            int(N_CLASSES),
    'benign_label':         int(BENIGN_LABEL),
    'train_macro_f1':       float(train_metrics['macro_f1']),
    'val_macro_f1':         float(val_metrics['macro_f1']),
    'test_macro_f1':        float(final_test_metrics['macro_f1']),
    'train_log_loss':       None,
    'val_log_loss':         None,
    'test_log_loss':        None,
    'test_benign_fpr':      float(final_test_metrics['benign_fpr']),
    'test_per_class_recall': final_test_metrics['per_class_recall'].tolist(),
    'test_accuracy':        float(final_test_metrics['accuracy']),
    'test_macro_precision': float(final_test_metrics['macro_prec']),
    'test_macro_recall':    float(final_test_metrics['macro_rec']),
}

metrics_path = save_dir / f'perceptron_metrics_{FEATURE_VERSION}.json'
with open(metrics_path, 'w') as f:
    json.dump(metrics_record, f, indent=2)

cm_path = save_dir / f'perceptron_cm_{FEATURE_VERSION}.npy'
np.save(cm_path, final_test_metrics['confusion_matrix'])

print(f'Saved metrics to:          {metrics_path}')
print(f'Saved confusion matrix to: {cm_path}')

## 13. Compare Variants

Load all saved Perceptron metric files and compare the trained variants in one table.

The code below collects all saved Perceptron runs into a comparison table.

In [ ]:
variant_files = sorted(save_dir.glob('perceptron_metrics_*.json'))

if not variant_files:
    print('No metrics files found yet. Re-run the notebook with a FEATURE_VERSION to populate this table.')
else:
    rows = []
    for f in variant_files:
        rec = json.loads(f.read_text())
        bp = rec['best_params']
        rows.append({
            'variant':              rec['variant'],
            'penalty':              bp.get('penalty'),
            'alpha':                bp.get('alpha'),
            'train_f1':             rec['train_macro_f1'],
            'val_f1':               rec['val_macro_f1'],
            'test_f1':              rec['test_macro_f1'],
            'train_val_gap':        rec['train_macro_f1'] - rec['val_macro_f1'],
            'test_accuracy':        rec.get('test_accuracy'),
            'test_macro_precision': rec.get('test_macro_precision'),
            'test_macro_recall':    rec.get('test_macro_recall'),
            'test_benign_fpr':      rec['test_benign_fpr'],
            'n_iter':               rec['n_iter'],
        })

    comp = pd.DataFrame(rows).sort_values('test_f1', ascending=False).reset_index(drop=True)
    print(f'Comparing {len(comp)} variant(s) of Perceptron:\n')
    print(comp.to_string(index=False))

    comp_path = save_dir / 'perceptron_comparison.csv'
    comp.to_csv(comp_path, index=False)
    print(f'\nSaved comparison table to: {comp_path}')